Ground truth log:

In [21]:
import subprocess, sys, json

_CHILD = r'''
import sys, json, tkinter as tk
from tkinter import filedialog
opts = json.loads(sys.argv[1])
root = tk.Tk(); root.withdraw()
try: root.attributes("-topmost", True)
except Exception: pass
p = filedialog.askopenfilename(title=opts["title"], initialdir=opts["initialdir"],
                               filetypes=[tuple(ft) for ft in opts["filetypes"]])
root.destroy()
sys.stdout.write(p or "")
'''

def pick_file(title="Select a file", initialdir=".",
              filetypes=(("OCEL logs", "*.json *.jsonocel"), ("All files", "*.*"))):
    """Runs the Tk dialog in a child process: if Tk dies, only the child dies."""
    opts = {"title": title, "initialdir": initialdir,
            "filetypes": [list(ft) for ft in filetypes]}
    r = subprocess.run([sys.executable, "-c", _CHILD, json.dumps(opts)],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(r.stderr.strip() or f"file picker exited {r.returncode}")
    return r.stdout.strip() or None

def ensure_ocel2(path, out_dir=None, force=False):
    """Accept OCEL 2.0 or OCEL 1.0 (.jsonocel). A 2.0 file is returned unchanged; a 1.0 file
       (ocel:events / ocel:objects, e.g. BPIC17.jsonocel) is converted once to a 2.0-shaped
       file and that path is returned, so every measure cell reads one structure.
       OCEL 1.0 carries no object-to-object relations: objectRelations is emitted empty, which
       no measure here reads. Attributes are dropped — the measures use timestamps only."""
    from pathlib import Path
    import tempfile
    path = Path(path)
    raw = json.load(open(path, encoding="utf-8"))
    if "events" in raw and "objects" in raw:
        print(f"OCEL 2.0: {path.name}")
        return str(path)
    if "ocel:events" not in raw or "ocel:objects" not in raw:
        raise ValueError(f"{path.name} is neither OCEL 2.0 nor OCEL 1.0 "
                         f"(top-level keys: {sorted(raw)[:6]})")
    out = Path(out_dir or tempfile.gettempdir()) / (path.stem + "_ocel2.json")
    if out.exists() and not force and out.stat().st_mtime >= path.stat().st_mtime:
        print(f"OCEL 1.0: {path.name} -> reusing conversion at {out}")
        return str(out)
    objects = [{"id": str(oid), "type": o.get("ocel:type", ""), "attributes": []}
               for oid, o in raw["ocel:objects"].items()]
    events = [{"id": str(eid), "type": e["ocel:activity"], "time": e["ocel:timestamp"],
               "relationships": [{"objectId": str(o), "qualifier": ""} for o in e.get("ocel:omap", [])],
               "attributes": []}
              for eid, e in raw["ocel:events"].items()]
    events.sort(key=lambda e: e["time"])
    json.dump({"version": "2.0", "ordering": "timestamp",
               "objectTypes": [{"name": t, "attributes": []} for t in sorted({o["type"] for o in objects})],
               "eventTypes": [{"name": a, "attributes": []} for a in sorted({e["type"] for e in events})],
               "objects": objects, "events": events, "objectRelations": []},
              open(out, "w", encoding="utf-8"))
    print(f"OCEL 1.0: {path.name} -> converted {len(events):,} events / {len(objects):,} objects")
    print(f"          written to {out}")
    return str(out)

INPUT_LOG = pick_file("Select the input (ground truth) log — OCEL 2.0 .json or OCEL 1.0 .jsonocel")
if INPUT_LOG is None:
    raise SystemExit("no input log selected")
INPUT_LOG = ensure_ocel2(INPUT_LOG)      # pass out_dir=... to place the conversion elsewhere

Simulated event log:

In [22]:
import subprocess, sys, json

_CHILD = r'''
import sys, json, tkinter as tk
from tkinter import filedialog
opts = json.loads(sys.argv[1])
root = tk.Tk(); root.withdraw()
try: root.attributes("-topmost", True)
except Exception: pass
p = filedialog.askopenfilename(title=opts["title"], initialdir=opts["initialdir"],
                               filetypes=[tuple(ft) for ft in opts["filetypes"]])
root.destroy()
sys.stdout.write(p or "")
'''

def pick_file(title="Select a file", initialdir=".",
              filetypes=(("JSON files", "*.json"), ("All files", "*.*"))):
    """Runs the Tk dialog in a child process: if Tk dies, only the child dies."""
    opts = {"title": title, "initialdir": initialdir,
            "filetypes": [list(ft) for ft in filetypes]}
    r = subprocess.run([sys.executable, "-c", _CHILD, json.dumps(opts)],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(r.stderr.strip() or f"file picker exited {r.returncode}")
    return r.stdout.strip() or None

SIMULATED_LOG = pick_file("Select event log A")

OC-Declare Model file:

In [23]:
# ===== Select the OC-Declare model ==========================================
import json
from pathlib import Path

# start in the ocdeclare folder next to the chosen input log, if it exists
_default = Path(INPUT_LOG).parent.parent / "ocdeclare"
OCDECLARE_MODEL = pick_file("Select the OC-Declare model",
                            initialdir=str(_default if _default.exists() else Path(INPUT_LOG).parent),
                            filetypes=(("JSON files", "*.json"), ("All files", "*.*")))
if OCDECLARE_MODEL is None:
    raise SystemExit("no model selected")

def load_model(path, expect_activities=None):
    """Load an OC-Declare model and fail loudly if the file is actually an OCEL log —
       both are .json and they sit in sibling folders, so this is an easy mix-up."""
    path = Path(path)
    d = json.load(open(path))
    if "events" in d and "objects" in d:
        raise ValueError(f"{path.name} is an OCEL event log, not an OC-Declare model")
    if "constraints" not in d or "activities" not in d:
        raise ValueError(f"{path.name} has no 'constraints'/'activities' — not an OC-Declare model "
                         f"(top-level keys: {sorted(d)[:8]})")
    acts = sorted(a["name"] for a in d["activities"])
    print(f"model: {path.name}")
    print(f"  {len(d['constraints'])} constraints, {len(acts)} activities, "
          f"{len(d.get('object_types', []))} object types, {len(d.get('o2o_rules', []))} o2o rules")
    if expect_activities is not None:
        missing = set(acts) - set(expect_activities)
        if missing: print(f"  ! model activities absent from the input log: {sorted(missing)}")
        else:       print("  activity alphabet matches the input log")
    return d

model = load_model(OCDECLARE_MODEL)
# after log_in exists, you can check the pairing:
# model = load_model(OCDECLARE_MODEL, expect_activities={a for _, a, _, _ in log_in["events"]})

model: discovered_container_logistics_20260915_090557.json
  35 constraints, 14 activities, 7 object types, 9 o2o rules


In [24]:
# ===== Load the three inputs once (run after the file-selection cell) =======
import json, pandas as pd

def load_ocel(path):
    raw = json.load(open(path))
    ts = pd.to_datetime([e["time"] for e in raw["events"]], utc=True, format="ISO8601")
    ev = sorted(zip((e["id"] for e in raw["events"]), (e["type"] for e in raw["events"]), ts,
                    ([r["objectId"] for r in e.get("relationships") or []] for e in raw["events"])),
                key=lambda x: x[2])
    return {"events": ev, "otype": {o["id"]: o["type"] for o in raw["objects"]}, "raw": raw}

log_in  = load_ocel(INPUT_LOG)        # parsed OCEL, what every measure consumes
log_sim = load_ocel(SIMULATED_LOG)
model   = json.load(open(OCDECLARE_MODEL))

print(f"input:     {len(log_in['events']):>7,} events, {len(log_in['otype']):>6,} objects")
print(f"simulated: {len(log_sim['events']):>7,} events, {len(log_sim['otype']):>6,} objects")
print(f"model:     {len(model['constraints'])} constraints, {len(model['activities'])} activities")

input:      35,372 events, 13,882 objects
simulated:  29,938 events, 16,511 objects
model:     35 constraints, 14 activities


Simulated log difference from ground truth, events and objects:

KL Divergence events

In [25]:
# ============ M1 — KL divergence for activities (log comparison) ==============
# paths come from the input-files cell: INPUT_LOG, SIMULATED_LOG
import json, numpy as np, pandas as pd
from collections import Counter

def load_ocel(path):
    raw = json.load(open(path))
    ts = pd.to_datetime([e["time"] for e in raw["events"]], utc=True, format="ISO8601")
    ev = sorted(zip((e["id"] for e in raw["events"]), (e["type"] for e in raw["events"]), ts,
                    ([r["objectId"] for r in e.get("relationships") or []] for e in raw["events"])),
                key=lambda x: x[2])
    return {"events": ev, "otype": {o["id"]: o["type"] for o in raw["objects"]}, "raw": raw}

def kl_divergence(cP, cQ, smoothing=0.5):
    """D_KL(P||Q) = sum_x P(x) log2(P(x)/Q(x)); additive smoothing keeps it finite when a
       category is missing from Q (otherwise the divergence is infinite)."""
    keys = sorted(set(cP) | set(cQ))
    p = np.array([cP.get(k, 0)+smoothing for k in keys], float); p /= p.sum()
    q = np.array([cQ.get(k, 0)+smoothing for k in keys], float); q /= q.sum()
    return float(np.sum(p*np.log2(p/q)))

log_in, log_sim = load_ocel(INPUT_LOG), load_ocel(SIMULATED_LOG)
c_in  = Counter(a for _, a, _, _ in log_in["events"])
c_sim = Counter(a for _, a, _, _ in log_sim["events"])
m     = Counter({k: (c_in.get(k,0)+c_sim.get(k,0))/2 for k in set(c_in)|set(c_sim)})

kl_activities = {
    "n_activities": len(set(c_in)|set(c_sim)),
    "KL_input||simulated_bits": kl_divergence(c_in, c_sim),   # the one to report
    "KL_simulated||input_bits": kl_divergence(c_sim, c_in),
    "JS_bits": 0.5*kl_divergence(c_in, m) + 0.5*kl_divergence(c_sim, m),   # symmetric sanity check
    "only_in_input": sorted(set(c_in)-set(c_sim)),
    "only_in_simulated": sorted(set(c_sim)-set(c_in)),
}
per_activity = pd.DataFrame(
    [{"activity": a, "count_input": c_in.get(a,0), "count_simulated": c_sim.get(a,0),
      "p_input": c_in.get(a,0)/sum(c_in.values()), "p_simulated": c_sim.get(a,0)/sum(c_sim.values())}
     for a in sorted(set(c_in)|set(c_sim))])
per_activity["contribution_bits"] = per_activity.apply(
    lambda r: r.p_input*np.log2(r.p_input/r.p_simulated)
    if r.p_input > 0 and r.p_simulated > 0 else np.nan, axis=1)

from IPython.display import display

print(f"KL(input || simulated) = {kl_activities['KL_input||simulated_bits']:.4f} bits")
print(f"KL(simulated || input) = {kl_activities['KL_simulated||input_bits']:.4f} bits")
print(f"JS divergence          = {kl_activities['JS_bits']:.4f} bits   (symmetric, 0..1)")
print(f"{kl_activities['n_activities']} activities | only in input: {kl_activities['only_in_input']}"
      f" | only in simulated: {kl_activities['only_in_simulated']}")
display(per_activity.sort_values("contribution_bits", ascending=False))

KL(input || simulated) = 0.1582 bits
KL(simulated || input) = 0.1324 bits
JS divergence          = 0.0339 bits   (symmetric, 0..1)
14 activities | only in input: [] | only in simulated: []


,activity,count_input,count_simulated,p_input,p_simulated,contribution_bits
7,Load to Vehicle,1966,297,0.055581,0.009921,0.138179
6,Load Truck,10553,7392,0.298343,0.246910,0.081444
1,Bring to Loading Bay,1969,1099,0.055665,0.036709,0.033435
13,Weigh,1989,1203,0.056231,0.040183,0.027259
5,Drive to Terminal,1989,1205,0.056231,0.040250,0.027125
4,Depart,127,3,0.003590,0.000100,0.018538
12,Reschedule Container,36,100,0.001018,0.003340,-0.001745
11,Register Customer Order,600,588,0.016963,0.019641,-0.003587
0,Book Vehicles,594,587,0.016793,0.019607,-0.003754
3,Create Transport Document,594,587,0.016793,0.019607,-0.003754


KL Divergence for object types:

In [26]:
# ============ M2 — KL divergence for object types (log comparison) ============
import json, numpy as np, pandas as pd
from collections import Counter





def kl_divergence(cP, cQ, smoothing=0.5):
    keys = sorted(set(cP) | set(cQ))
    p = np.array([cP.get(k, 0)+smoothing for k in keys], float); p /= p.sum()
    q = np.array([cQ.get(k, 0)+smoothing for k in keys], float); q /= q.sum()
    return float(np.sum(p*np.log2(p/q)))

def type_counts(L, basis):
    """basis='objects' -> how many objects of each type exist;
       basis='events'  -> how many event-to-object relations each type carries."""
    if basis == "objects": return Counter(L["otype"].values())
    return Counter(L["otype"].get(o) for _, _, _, objs in L["events"] for o in objs)

A, B = load_ocel(INPUT_LOG), load_ocel(SIMULATED_LOG)
rows = []
for basis in ("objects", "events"):
    cA, cB = type_counts(A, basis), type_counts(B, basis)
    m = Counter({k: (cA.get(k,0)+cB.get(k,0))/2 for k in set(cA)|set(cB)})
    rows.append({"basis": basis, "n_types": len(set(cA)|set(cB)),
                 "KL_A||B_bits": kl_divergence(cA, cB), "KL_B||A_bits": kl_divergence(cB, cA),
                 "JS_bits": 0.5*kl_divergence(cA, m) + 0.5*kl_divergence(cB, m),
                 "only_in_A": sorted(set(cA)-set(cB)), "only_in_B": sorted(set(cB)-set(cA))})
kl_object_types = pd.DataFrame(rows)

cA, cB = type_counts(A, "objects"), type_counts(B, "objects")
per_type = pd.DataFrame([{"object_type": t, "n_A": cA.get(t,0), "n_B": cB.get(t,0),
                          "p_A": cA.get(t,0)/sum(cA.values()), "p_B": cB.get(t,0)/sum(cB.values())}
                         for t in sorted(set(cA)|set(cB))])

print(kl_object_types.to_string())
per_type

     basis  n_types  KL_A||B_bits  KL_B||A_bits   JS_bits only_in_A only_in_B
0  objects        7      0.232495      0.911637  0.077292        []        []
1   events        7      0.019243      0.016478  0.004478        []        []


,object_type,n_A,n_B,p_A,p_B
0,Container,1999,1954,0.143999,0.118345
1,Customer Order,600,588,0.043221,0.035613
2,Forklift,3,1099,0.000216,0.066562
3,Handling Unit,10553,10553,0.760193,0.639150
4,Transport Document,594,587,0.042789,0.035552
5,Truck,6,1205,0.000432,0.072982
6,Vehicle,127,525,0.009149,0.031797


Relevative Event Distribution:

In [55]:
# ===== RED — Relative Event Distribution ====================================
# needs INPUT_LOG and SIMULATED_LOG (paths) from the file-selection cell
import json, numpy as np, pandas as pd
from collections import Counter
from scipy.stats import wasserstein_distance

BIN_SECONDS = 3600.0          # hourly bins, as in the definition

# ---- unit-aware printing ----------------------------------------------------
def _unit_of(key):
    k = key.lower()
    if k.endswith("_coverage"):                                  return "fraction"
    if k.startswith("ngd"):                                      return "ratio01"
    if k.startswith("wmape") or k.endswith("_norm"):             return "ratio"
    if k.startswith("red") or k.endswith(("_hours", "_h")) or "_h_" in k: return "hours"
    if k.startswith(("transitions", "n_", "bins")) or k.endswith(("_obs", "_pairs")): return "count"
    return "value"

_SCALES = {"fraction": "[0-1], share",
           "ratio":    "0 % = identical, unbounded",
           "ratio01":  "[0-1], 0 = identical, 1 = no shared n-grams",
           "hours":    "[0, inf) hours, 0 = identical",
           "count":    "", "value": ""}

def _fmt(key, v):
    if not isinstance(v, (int, float, np.floating, np.integer)) or isinstance(v, bool): return str(v)
    if v != v: return "       —"
    u = _unit_of(key)
    if u in ("fraction", "ratio01"): return f"{v:10.4f}  [0-1]"
    if u == "ratio":    return f"{v*100:10.2f} %    "
    if u == "hours":    return f"{v:10.4f}  h    "
    if u == "count":    return f"{v:10.0f}       "
    if isinstance(v, (int, np.integer)): return f"{v:10d}       "
    return f"{v:10.4f}       "

def print_result(d, title=None):
    if title: print(title)
    for k, v in d.items():
        scale = _SCALES.get(_unit_of(k), "") if isinstance(v, (int, float, np.floating, np.integer)) else ""
        print(f"  {k:26}{_fmt(k, v)}   {scale}")

def format_table(df):
    out = df.copy()
    for c in out.columns:
        if out[c].dtype.kind in "fi": out[c] = [_fmt(c, x).strip() for x in out[c]]
    return out

# ---- measure ----------------------------------------------------------------
def _load(path):
    raw = json.load(open(path))
    otype = {o["id"]: o["type"] for o in raw["objects"]}
    evs = sorted(((e["time"], e["type"], [r["objectId"] for r in e.get("relationships") or []])
                  for e in raw["events"] if e.get("time")), key=lambda x: x[0])
    ts = pd.to_datetime([e[0] for e in evs], utc=True, format="ISO8601")
    return {"events": [(t, e[1], e[2]) for t, e in zip(ts, evs)], "otype": otype}

def relative_event_times(L, object_type=None):
    """rho(e) = tau(e) - a(xi(e)).  OCEL has no case notion, so the 'case arrival' a(.) is read
       object-centrically: the first event timestamp of the object the event is offset against.
       An event contributes one value per attached object of the chosen type."""
    birth, rho = {}, []
    for t, act, objs in L["events"]:                       # already in timestamp order
        for o in objs:
            if object_type is not None and L["otype"].get(o) != object_type: continue
            if o not in birth: birth[o] = t
            rho.append((t - birth[o]).total_seconds())
    return np.asarray(rho, float)

def red_distance(LA, LB, object_type=None, bin_s=BIN_SECONDS):
    """RED = EMD between the two logs' hourly-discretised rho(e). Value is in bins (= hours)."""
    ra = np.floor(relative_event_times(LA, object_type)/bin_s)
    rb = np.floor(relative_event_times(LB, object_type)/bin_s)
    if not len(ra) or not len(rb):
        return {"object_type": object_type or "<all>", "n_input": len(ra), "n_simulated": len(rb),
                "RED_hours": np.nan}
    ca, cb = Counter(ra), Counter(rb)
    va, wa = np.array(sorted(ca)), np.array([ca[k] for k in sorted(ca)], float)
    vb, wb = np.array(sorted(cb)), np.array([cb[k] for k in sorted(cb)], float)
    return {"object_type": object_type or "<all>", "n_input": len(ra), "n_simulated": len(rb),
            "bins_input": len(ca), "bins_simulated": len(cb),
            "mean_h_input": float(np.average(va, weights=wa)),
            "mean_h_simulated": float(np.average(vb, weights=wb)),
            "RED_hours": float(wasserstein_distance(va, vb, wa, wb))}

if "log_in_red" not in globals():
    log_in_red, log_sim_red = _load(INPUT_LOG), _load(SIMULATED_LOG)

_types = sorted(set(log_in_red["otype"].values()) | set(log_sim_red["otype"].values()))
red_table = pd.DataFrame([red_distance(log_in_red, log_sim_red, t) for t in _types]
                         + [red_distance(log_in_red, log_sim_red, None)])
red_result = {"bin_width_hours": BIN_SECONDS/3600,
              "RED_pooled_hours": red_table.iloc[-1]["RED_hours"],
              "RED_worst_type_hours": float(red_table.iloc[:-1]["RED_hours"].max()),
              "RED_mean_over_types_hours": float(red_table.iloc[:-1]["RED_hours"].mean())}
print_result(red_result, "RED — relative event distribution (offset from each object's first event)")
print()
format_table(red_table)

RED — relative event distribution (offset from each object's first event)
  bin_width_hours               1.0000  h       [0, inf) hours, 0 = identical
  RED_pooled_hours           1298.7900  h       [0, inf) hours, 0 = identical
  RED_worst_type_hours       5394.5789  h       [0, inf) hours, 0 = identical
  RED_mean_over_types_hours  1696.5551  h       [0, inf) hours, 0 = identical



,object_type,n_input,n_simulated,bins_input,bins_simulated,mean_h_input,mean_h_simulated,RED_hours
0,Container,26266,18993,573,2449,153.5153 h,436.0298 h,410.0578 h
1,Customer Order,1194,1175,189,61,45.2186 h,3.6221 h,41.5965 h
2,Forklift,7738,6384,2516,32,5396.9390 h,2.3601 h,5394.5789 h
3,Handling Unit,21106,17945,38,1854,0.9738 h,255.4551 h,254.4813 h
4,Transport Document,2571,1864,323,63,103.4807 h,10.8197 h,95.4328 h
5,Truck,12542,8597,3707,33,5285.9165 h,0.3901 h,5285.5264 h
6,Vehicle,2855,991,467,106,428.9832 h,42.0757 h,394.2122 h
7,<all>,74272,55949,5271,3359,1530.2544 h,231.4643 h,1298.7900 h


Time comparison:

Time distribution on log

In [ ]:
# ===== Temporal transition matrix, discovered per object type ================
import json, statistics
from pathlib import Path
from collections import defaultdict
from datetime import datetime

def object_traces(log_path):
    """One trace per OBJECT (not per 'case'), grouped by object type. An OCEL has no single
       case notion: picking one type as the case throws away every event that type never
       touches. Returns {object_type: {object_id: [(activity, timestamp), ...]}}."""
    data = json.load(open(log_path, encoding="utf-8"))
    if not (isinstance(data, dict) and "events" in data and "objects" in data):
        raise ValueError(f"{Path(log_path).name} is not an OCEL 2.0 log with events and objects")
    otype = {o["id"]: o["type"] for o in data["objects"]}
    traces = defaultdict(lambda: defaultdict(list))        # inverted index: one pass, not N passes
    for ev in data["events"]:
        ts = ev.get("time")
        if not ts: continue
        t = datetime.fromisoformat(ts.replace("Z", "+00:00"))
        for rel in ev.get("relationships") or []:
            oid = rel["objectId"]
            traces[otype.get(oid)][oid].append((ev["type"], t))
    for ot in traces:
        for oid in traces[ot]: traces[ot][oid].sort(key=lambda x: x[1])
    return traces

def _stats(deltas, keep_deltas=True):
    s = {"count": len(deltas), "mean_seconds": statistics.mean(deltas),
         "median_seconds": statistics.median(deltas),
         "std_dev_seconds": statistics.stdev(deltas) if len(deltas) > 1 else 0.0,
         "min_seconds": min(deltas), "max_seconds": max(deltas)}
    if len(deltas) >= 4:
        q = statistics.quantiles(deltas, n=4)
        s["percentile_25_seconds"], s["percentile_75_seconds"] = q[0], q[2]
    else:
        s["percentile_25_seconds"], s["percentile_75_seconds"] = s["min_seconds"], s["max_seconds"]
    if keep_deltas: s["time_deltas_seconds"] = deltas   # needed for Wasserstein-1 / WMAPE later
    return s

def discover_temporal_matrix(log_path, keep_deltas=True):
    """{object_type: {source_activity: {target_activity: stats}}} — directly-follows waiting
       times computed inside each object's own timeline."""
    matrix, covered = {}, set()
    for ot, objs in object_traces(log_path).items():
        deltas = defaultdict(lambda: defaultdict(list))
        for oid, trace in objs.items():
            for i in range(len(trace)-1):
                (sa, st), (ta, tt) = trace[i], trace[i+1]
                deltas[sa][ta].append((tt-st).total_seconds())
                covered.add(sa); covered.add(ta)
        matrix[ot] = {sa: {ta: _stats(v, keep_deltas) for ta, v in tg.items()}
                      for sa, tg in deltas.items()}
    matrix["_meta"] = {"log": str(log_path), "object_types": sorted(k for k in matrix if k != "_meta"),
                       "activities_covered": sorted(covered),
                       "n_transitions": sum(len(tg) for ot, m in matrix.items() if ot != "_meta"
                                            for tg in m.values())}
    return matrix

# run it for both logs and KEEP the results
TEMPORAL = {"input":     discover_temporal_matrix(INPUT_LOG),
            "simulated": discover_temporal_matrix(SIMULATED_LOG)}
for role, m in TEMPORAL.items():
    print(f"{role:10} {m['_meta']['n_transitions']:3} transitions, "
          f"{len(m['_meta']['object_types'])} object types, "
          f"{len(m['_meta']['activities_covered'])} activities")

input       49 transitions, 7 object types, 14 activities
simulated   86 transitions, 7 object types, 14 activities


WMAPE and Waserstein-1 between both logs:

WMAPE:

In [71]:
# ===== WMAPE on the discovered service times ================================
# Standard WMAPE (MAD/Mean ratio), no external weights:
#     WMAPE = sum_b |A_b - F_b| / sum_b |A_b|
# A_b = service time of activity b in the INPUT log, F_b = in the SIMULATED log.
# Input: TEMPORAL from the temporal discovery cell — no log is re-read.
import numpy as np, pandas as pd

if "TEMPORAL" not in globals():
    raise NameError("TEMPORAL is missing — run the temporal discovery cell first")

def service_times(matrix):
    """{activity: (mean_h, std_h)} — every transition ENDING in an activity is that activity's
       service time. Pooled mean and pooled std are exact from the stored count/mean/std."""
    groups = {}
    for ot, mm in matrix.items():
        if ot == "_meta": continue
        for sa, tg in mm.items():
            for ta, st in tg.items():
                groups.setdefault(ta, []).append((st["count"], st["mean_seconds"]/3600,
                                                  st["std_dev_seconds"]/3600))
    out = {}
    for act, parts in groups.items():
        n = sum(p[0] for p in parts)
        if n == 0: continue
        mu = sum(p[0]*p[1] for p in parts)/n
        var = (sum((p[0]-1)*p[2]**2 for p in parts if p[0] > 1)
               + sum(p[0]*(p[1]-mu)**2 for p in parts))/(n-1) if n > 1 else 0.0
        out[act] = (mu, float(np.sqrt(var)))
    return out

A, F = service_times(TEMPORAL["input"]), service_times(TEMPORAL["simulated"])
shared = sorted(set(A) & set(F))

service = pd.DataFrame([{"activity": a,
                         "mean_h_input": A[a][0], "mean_h_simulated": F[a][0],
                         "std_h_input": A[a][1],  "std_h_simulated": F[a][1]} for a in shared])

wmape_mean = (service.mean_h_input - service.mean_h_simulated).abs().sum() / service.mean_h_input.abs().sum()
wmape_std  = (service.std_h_input  - service.std_h_simulated).abs().sum()  / service.std_h_input.abs().sum()

print("WMAPE on discovered service times   (input = actual A, simulated = forecast F)")
print(f"  activities compared    {len(shared)} of {len(set(A) | set(F))}")
print(f"  WMAPE service mean     {wmape_mean*100:8.2f} %")
print(f"  WMAPE service std      {wmape_std*100:8.2f} %")
print()
print(service.to_string(index=False, float_format=lambda v: f"{v:9.2f}"))

WMAPE on discovered service times   (input = actual A, simulated = forecast F)
  activities compared    12 of 12
  WMAPE service mean       194.03 %
  WMAPE service std        342.42 %

                 activity  mean_h_input  mean_h_simulated  std_h_input  std_h_simulated
            Book Vehicles         43.24              5.44        69.81            11.38
     Bring to Loading Bay         98.09            272.99       131.42           544.16
Create Transport Document         91.39              7.71        77.96            13.34
                   Depart         90.73              5.62       140.67             1.46
        Drive to Terminal          3.06              0.63         8.69             3.77
               Load Truck          3.66            219.04        11.70           778.84
          Load to Vehicle          2.68             60.84        15.84           182.78
   Order Empty Containers         12.20              3.19        17.02             5.69
  Pick Up Empty Contai

Wasserstein-1

In [ ]:
# ===== Wasserstein-1 on the discovered temporal matrices ===================== WRONG NOT USING DISCOVERED MATRIX AGAIN
# needs TEMPORAL from the discovery cell (with keep_deltas=True, the default)
import numpy as np, pandas as pd
from scipy.stats import wasserstein_distance

if "TEMPORAL" not in globals():
    raise NameError("TEMPORAL is missing — run the temporal discovery cell first")

def _flatten(matrix):
    return {(ot, sa, ta): st
            for ot, mm in matrix.items() if ot != "_meta"
            for sa, tg in mm.items() for ta, st in tg.items()}

def w1_frame(TEMPORAL, min_n=5):
    """W1(P,Q) = integral |F_P - F_Q| dx between the two logs' raw waiting-time samples, in
       hours. W1 >= |mean difference| always, so shape_excess is disagreement in spread/shape
       that a comparison of means cannot see."""
    fi, fs = _flatten(TEMPORAL["input"]), _flatten(TEMPORAL["simulated"])
    rows = []
    for k in sorted(set(fi) & set(fs)):
        if "time_deltas_seconds" not in fi[k]:
            raise KeyError("raw deltas missing — rerun discovery with keep_deltas=True")
        xa = np.asarray(fi[k]["time_deltas_seconds"], float)/3600.0
        xb = np.asarray(fs[k]["time_deltas_seconds"], float)/3600.0
        if len(xa) < min_n or len(xb) < min_n: continue
        w1, gap = wasserstein_distance(xa, xb), abs(xa.mean()-xb.mean())
        rows.append({"object_type": k[0], "source": k[1], "target": k[2],
                     "n_input": len(xa), "n_simulated": len(xb),
                     "mean_h_input": xa.mean(), "mean_h_simulated": xb.mean(),
                     "W1_hours": w1, "W1_norm": w1/xa.mean() if xa.mean() else np.nan,
                     "mean_gap_h": gap, "shape_excess_h": w1-gap})
    return pd.DataFrame(rows)

def w1_aggregate(wf, weight="n_input", normalised=False):
    col = "W1_norm" if normalised else "W1_hours"
    v, w = wf[col].values, wf[weight].values.astype(float)
    m = np.isfinite(v)
    return float(np.sum(v[m]*w[m])/np.sum(w[m])) if m.any() and np.sum(w[m]) else np.nan

def w1_per_type(wf):
    return (pd.DataFrame([{"object_type": ot, "transitions": len(g),
                           "n_input_obs": int(g.n_input.sum()),
                           "W1_hours": w1_aggregate(g),
                           "W1_norm": w1_aggregate(g, normalised=True)}
                          for ot, g in wf.groupby("object_type")])
            .sort_values("n_input_obs", ascending=False))

wf = w1_frame(TEMPORAL)
_fi, _fs = _flatten(TEMPORAL["input"]), _flatten(TEMPORAL["simulated"])
w1_result = {
    "transitions_scored": len(wf), "transitions_total": len(set(_fi) | set(_fs)),
    "observation_coverage": wf.n_input.sum()/sum(s["count"] for s in _fi.values()),
    "W1_weighted_hours": w1_aggregate(wf),
    "W1_weighted_norm": w1_aggregate(wf, normalised=True),
    "W1_unweighted_hours": float(wf.W1_hours.mean()),
    "W1_median_hours": float(wf.W1_hours.median()),
}
for k, v in w1_result.items():
    if not isinstance(v, float):        print(f"{k:24} {v:>10}")
    elif k.endswith("_norm"):           print(f"{k:24} {v*100:>10.2f} %       ratio, 0 % = identical")
    elif k.endswith("_hours"):          print(f"{k:24} {v:>10.4f} h       [0, inf) hours")
    elif k.endswith("_coverage"):       print(f"{k:24} {v:>10.4f}  [0-1]")
    else:                               print(f"{k:24} {v:>10.4f}")
print("\nper object type:"); print(w1_per_type(wf).to_string(index=False))
print("\ntop 5 by absolute disagreement:")
print(wf.nlargest(5, "W1_hours")[["object_type","source","target","n_input",
                                  "mean_h_input","mean_h_simulated","W1_hours"]].to_string(index=False))
print("\ntop 5 by shape excess (means agree, distributions do not):")
wf.nlargest(5, "shape_excess_h")[["object_type","source","target","n_input",
                                  "mean_gap_h","W1_hours","shape_excess_h"]]

transitions_scored       36
transitions_total        95
observation_coverage     0.8871
W1_weighted_hours        152.9446
W1_weighted_norm         75.6323
W1_unweighted_hours      58.0795
W1_median_hours          8.2510

per object type:
       object_type  transitions  n_input_obs   W1_hours    W1_norm
         Container           11        22311  67.093576  43.970016
     Handling Unit            1        10553 618.460454 284.679515
             Truck            2        10548   1.599268   0.931599
          Forklift           11         5767   5.239452   0.950069
           Vehicle            7         2599  25.173860  13.909456
Transport Document            3         1200   6.493480   0.597661
    Customer Order            1          594  83.678099   0.915612

top 5 by absolute disagreement:
  object_type                 source                  target  n_input  mean_h_input  mean_h_simulated   W1_hours
Handling Unit          Collect Goods              Load Truck    10553      2.172

,object_type,source,target,n_input,mean_gap_h,W1_hours,shape_excess_h
26,Transport Document,Order Empty Containers,Reschedule Container,13,24.629141,78.654671,54.025531
34,Vehicle,Reschedule Container,Load to Vehicle,7,32.218209,61.933627,29.715418
8,Container,Reschedule Container,Load to Vehicle,35,25.734942,53.476785,27.741844
30,Vehicle,Book Vehicles,Load to Vehicle,122,45.854706,57.565858,11.711152
7,Container,Place in Stock,Bring to Loading Bay,1794,359.484870,371.181491,11.696621


NGD:

In [72]:

# ===== NGD — N-Gram Distance ================================================
# needs INPUT_LOG and SIMULATED_LOG (paths) from the file-selection cell
import json, numpy as np, pandas as pd
from collections import Counter, defaultdict

NS = (2, 3)        # 2-grams = DFG histogram, 3-grams = 2nd-order abstraction

# ---- unit-aware printing ----------------------------------------------------
def _unit_of(key):
    k = key.lower()
    if k.endswith("_coverage"):                                  return "fraction"
    if k.startswith("ngd"):                                      return "ratio01"
    if k.startswith("wmape") or k.endswith("_norm"):             return "ratio"
    if k.startswith("red") or k.endswith(("_hours", "_h")) or "_h_" in k: return "hours"
    if k.startswith(("n_", "grams", "transitions")) or k.endswith(("_obs", "_grams")): return "count"
    return "value"

_SCALES = {"fraction": "[0-1], share",
           "ratio":    "0 % = identical, unbounded",
           "ratio01":  "[0-1], 0 = identical, 1 = no shared n-grams",
           "hours":    "[0, inf) hours, 0 = identical",
           "count":    "", "value": ""}

def _fmt(key, v):
    if not isinstance(v, (int, float, np.floating, np.integer)) or isinstance(v, bool): return str(v)
    if v != v: return "       —"
    u = _unit_of(key)
    if u in ("fraction", "ratio01"): return f"{v:10.4f}  [0-1]"
    if u == "ratio":    return f"{v*100:10.2f} %    "
    if u == "hours":    return f"{v:10.4f}  h    "
    if u == "count":    return f"{v:10.0f}       "
    if isinstance(v, (int, np.integer)): return f"{v:10d}       "
    return f"{v:10.4f}       "

def print_result(d, title=None):
    if title: print(title)
    for k, v in d.items():
        scale = _SCALES.get(_unit_of(k), "") if isinstance(v, (int, float, np.floating, np.integer)) else ""
        print(f"  {k:26}{_fmt(k, v)}   {scale}")

def format_table(df):
    out = df.copy()
    for c in out.columns:
        if out[c].dtype.kind in "fi": out[c] = [_fmt(c, x).strip() for x in out[c]]
    return out

# ---- measure ----------------------------------------------------------------
def _load(path):
    raw = json.load(open(path))
    otype = {o["id"]: o["type"] for o in raw["objects"]}
    evs = sorted(((e["time"], e["type"], [r["objectId"] for r in e.get("relationships") or []])
                  for e in raw["events"] if e.get("time")), key=lambda x: x[0])
    return {"events": [(e[0], e[1], e[2]) for e in evs], "otype": otype}

def ngrams_of_log(L, object_type=None, n=2, dummy="0"):
    """Cases = objects (of one type), temporal information abstracted away. Each activity
       sequence is padded with n-1 dummy activities at both ends, so every activity instance
       appears in exactly n n-grams (otherwise first/last instances would count only once)."""
    seq = defaultdict(list)
    for t, act, objs in L["events"]:                       # already in timestamp order
        for o in objs:
            if object_type is not None and L["otype"].get(o) != object_type: continue
            seq[o].append(act)
    c = Counter()
    for o, s in seq.items():
        s = [dummy]*(n-1) + s + [dummy]*(n-1)
        for i in range(len(s)-n+1): c[tuple(s[i:i+n])] += 1
    return c

def ngd(LA, LB, object_type=None, n=2):
    """Relative NGD = Total Variation Distance between normalized n-gram frequencies.
       0 = identical relative distributions, 1 = no shared n-grams."""
    cA, cB = ngrams_of_log(LA, object_type, n), ngrams_of_log(LB, object_type, n)
    NA, NB = sum(cA.values()), sum(cB.values())

    # Cannot normalize if either distribution is empty.
    if NA == 0 or NB == 0:
        return {"object_type": object_type or "<all>", "n": n, "NGD_relative": np.nan}

    keys = set(cA) | set(cB)
    rel = 0.5*sum(abs(cA.get(g, 0)/NA - cB.get(g, 0)/NB) for g in keys)

    return {"object_type": object_type or "<all>", "n": n,
            "NGD_relative": rel}

if "log_in_ngd" not in globals():
    log_in_ngd, log_sim_ngd = _load(INPUT_LOG), _load(SIMULATED_LOG)

_types = sorted(set(log_in_ngd["otype"].values()) | set(log_sim_ngd["otype"].values()))
ngd_table = pd.DataFrame([ngd(log_in_ngd, log_sim_ngd, t, n) for n in NS for t in _types]
                         + [ngd(log_in_ngd, log_sim_ngd, None, n) for n in NS])

ngd_result = {}
for n in NS:
    pooled = ngd_table[(ngd_table.n == n) & (ngd_table.object_type == "<all>")].iloc[0]
    ngd_result[f"NGD_{n}gram_relative"] = float(pooled["NGD_relative"])

print_result(ngd_result, "NGD — relative n-gram distance (2-grams and 3-grams)")
print()
format_table(ngd_table)

NGD — relative n-gram distance (2-grams and 3-grams)
  NGD_2gram_relative            0.3101  [0-1]   [0-1], 0 = identical, 1 = no shared n-grams
  NGD_3gram_relative            0.3745  [0-1]   [0-1], 0 = identical, 1 = no shared n-grams



,object_type,n,NGD_relative
0,Container,2,0.3123 [0-1]
1,Customer Order,2,0.0037 [0-1]
2,Forklift,2,0.8884 [0-1]
3,Handling Unit,2,0.1479 [0-1]
4,Transport Document,2,0.6035 [0-1]
5,Truck,2,0.2453 [0-1]
6,Vehicle,2,0.7302 [0-1]
7,Container,3,0.4165 [0-1]
8,Customer Order,3,0.0047 [0-1]
9,Forklift,3,0.9646 [0-1]


Simulated Log vs Model:

OC-Declare Confidence Score:

In [74]:

# ===== OC-Declare confidence score (Definitions 8 & 9) ======================
# needs INPUT_LOG, SIMULATED_LOG, OCDECLARE_MODEL (paths) from the selection cells
import json, itertools, numpy as np, pandas as pd
from collections import defaultdict

# Definition 8: activation events of s; satisfaction is nmin <= |f(E_L)|.
EP_ACTIVATES_ON_TARGET = True     # OCDeclarediscovery writes EP arcs swapped:
                                  # "EP/DP(A->B,T): precedence(source=B, target=A)",
                                  # so Definition 8's activation s is target_activity.
# nmax is deliberately NOT enforced: the "nmax" field written by OCDeclarediscovery is
# the maximum occurrences of an activity PER SCOPE OBJECT (see _arcs_to_deco_constraints:
# "target may occur at most this often"), not Definition 8's bound on |f(E_L)|. All 35
# constraints carry nmin=1, so "nmin <= |f(E_L)|" is Definition 8 in full on that side.

# ---- unit-aware printing ----------------------------------------------------
def _unit_of(key):
    k = key.lower()
    if k.startswith(("conf", "gconf")) or k.endswith(("_confidence", "_rate", "_coverage")): return "fraction"
    if k.startswith("wmape") or k.endswith("_norm"):             return "ratio"
    if k.endswith(("_hours", "_h")) or "_h_" in k:               return "hours"
    if k.startswith(("n_", "events", "constraints")) or k.endswith(("_obs", "_events", "_count")): return "count"
    return "value"

_SCALES = {"fraction": "[0-1], 1 = fully conforming",
           "ratio": "0 % = identical, unbounded",
           "hours": "[0, inf) hours", "count": "", "value": ""}

def _fmt(key, v):
    if not isinstance(v, (int, float, np.floating, np.integer)) or isinstance(v, bool): return str(v)
    if v != v: return "       —"
    u = _unit_of(key)
    if u == "fraction": return f"{v:10.4f}  [0-1]"
    if u == "ratio":    return f"{v*100:10.2f} %    "
    if u == "hours":    return f"{v:10.4f}  h"
    if u == "count":    return f"{v:10.0f}"
    if isinstance(v, (int, np.integer)): return f"{v:10d}"
    return f"{v:10.4f}"

def print_result(d, title=None):
    if title: print(title)
    for k, v in d.items():
        scale = _SCALES.get(_unit_of(k), "") if isinstance(v, (int, float, np.floating, np.integer)) else ""
        print(f"  {k:26}{_fmt(k, v)}   {scale}")

def format_table(df):
    out = df.copy()
    for c in out.columns:
        if out[c].dtype.kind in "fi": out[c] = [_fmt(c, x).strip() for x in out[c]]
    return out

# ---- index ------------------------------------------------------------------
def build_index(path):
    raw = json.load(open(path))
    otype = {o["id"]: o["type"] for o in raw["objects"]}
    ev_objs, obj_events = defaultdict(lambda: defaultdict(set)), defaultdict(set)
    ev_time, act_events = {}, defaultdict(set)
    ts = pd.to_datetime([e["time"] for e in raw["events"]], utc=True, format="ISO8601")
    for e, t in zip(raw["events"], ts):
        ev_time[e["id"]] = t; act_events[e["type"]].add(e["id"])
        for r in e.get("relationships") or []:
            o = r["objectId"]; ev_objs[e["id"]][otype.get(o)].add(o); obj_events[o].add(e["id"])
    return {"ev_objs": ev_objs, "obj_events": obj_events, "ev_time": ev_time,
            "act_events": act_events, "otype": otype, "n_events": len(raw["events"])}

# ---- Definition 8 -----------------------------------------------------------
def constraint_roles(c, ep_on_target=EP_ACTIVATES_ON_TARGET):
    """D = (ar, s, t, oi, nmin, nmax).

       Evaluate events of source activity s, as specified in Definition 8.

       EF: count later events of target activity t.
       EP: count earlier events of target activity t.

       The EF/EP temporal directions must agree with the paper's
       earlier definitions and the model JSON's arc conventions.
    """
    ar = c.get("arc_type", "EF").upper()
    s, t = c["source_activity"], c["target_activity"]
    if ar == "EP" and ep_on_target: return t, s, "before", ar
    return (s, t, "after", ar) if ar == "EF" else (s, t, "before", ar)

def involvement(c):
    """Each_oi / All_oi / Any_oi from the model's involvement_per_label."""
    inv = c.get("involvement_per_label") or {lab: "each" for lab in c.get("label", [])}
    low = {k: str(v).lower() for k, v in inv.items()}
    return ([k for k, v in low.items() if v == "each"],
            [k for k, v in low.items() if v == "all"],
            [k for k, v in low.items() if v == "any"])

def f_filter(idx, e, act_t, direction, each_combo, all_objs, any_types):
    """|f(E_L)| of Definition 8: filter^ACT_t o filter^TIME_{time(e),ar} o filter^ALL_{o1..on}
       o filter^ALL_{All_oi objects of e} o filter^ANY_{Any_oi objects of e}."""
    required = set(each_combo) | all_objs
    if required:
        sets = sorted((idx["obj_events"][o] for o in required), key=len)
        cands = set(sets[0])
        for s in sets[1:]: cands &= s
        cands &= idx["act_events"][act_t]
    else:
        cands = set(idx["act_events"][act_t])
    te = idx["ev_time"][e]
    cands = {x for x in cands
             if (idx["ev_time"][x] > te if direction == "after" else idx["ev_time"][x] < te)}
    for ot in any_types:
        # Definition 8 applies filter^ANY unconditionally: if e has no object of this
        # type the filter admits nothing, f(E_L) is empty and the activation is violated.
        src = idx["ev_objs"][e].get(ot, set())
        cands = {x for x in cands if idx["ev_objs"][x].get(ot, set()) & src}
    return cands

def satisfies(idx, e, c):
    """Definition 8:
       e |= D iff for EVERY combination of EACH objects,
       nmin <= |f(E_L)| <= nmax.

       A quantifier over an empty object set is vacuously true.
    """
    act_s, act_t, direction, ar = constraint_roles(c)
    each, alls, anys = involvement(c)

    # Enforce BOTH cardinality bounds from Definition 8.
    nmin = c.get("nmin", 1)
    nmax = float("inf")                     # see the note at the top of the cell

    objs = idx["ev_objs"][e]
    all_objs = set().union(*[objs.get(ot, set()) for ot in alls]) if alls else set()
    domains = [sorted(objs.get(ot, set())) for ot in each]
    counts, ok = [], True

    # a universal quantifier over an empty Each-domain is vacuously true (Def. 8)
    for combo in (itertools.product(*domains) if domains else [()]):
        n = len(f_filter(idx, e, act_t, direction, combo, all_objs, anys))
        counts.append(n)
        if not (nmin <= n <= nmax): ok = False

    return ok, counts

# ---- Definition 9 -----------------------------------------------------------
def confidence_report(idx, model):
    """conf_L(D) = |{e in E_L^s : e |= D}| / |E_L^s|  in [0,1]."""
    rows = []
    for c in model["constraints"]:
        act_s, act_t, direction, ar = constraint_roles(c)
        each, alls, anys = involvement(c)
        events = idx["act_events"][act_s]
        sat, all_counts = 0, []
        for e in events:
            ok, counts = satisfies(idx, e, c)
            sat += ok; all_counts += counts
        rows.append({"constraint": f'{c.get("type")}({c["source_activity"]} -> {c["target_activity"]})',
                     "arc": ar, "activates_on": act_s, "counts_activity": act_t,
                     "each": "+".join(each), "any": "+".join(anys),
                     "nmin": c.get("nmin", 1), "nmax": c.get("nmax", 1),
                     "n_activations": len(events), "n_satisfied": sat,
                     "confidence": sat/len(events) if events else np.nan,
                     "mean_count": float(np.mean(all_counts)) if all_counts else np.nan,
                     "model_support": c.get("support", np.nan)})
    return pd.DataFrame(rows)

def global_confidence(idx, model):
    """gconf_L(DS) = |{e in E_L : for all D in DS, e |= D}| / |E_L|; events of a
       non-activating activity satisfy trivially (Definition 8)."""
    bad = set()
    for c in model["constraints"]:
        act_s, _, _, _ = constraint_roles(c)
        for e in idx["act_events"][act_s]:
            if e not in bad and not satisfies(idx, e, c)[0]: bad.add(e)
    n = idx["n_events"]
    return (n - len(bad))/n if n else np.nan, len(bad), n

# ---- load and calculate -----------------------------------------------------
model = json.load(open(OCDECLARE_MODEL))

idx_in, idx_sim = build_index(INPUT_LOG), build_index(SIMULATED_LOG)

conf_in = confidence_report(idx_in, model)
conf_sim = confidence_report(idx_sim, model)

g_in, bad_in, n_in = global_confidence(idx_in, model)
g_sim, bad_sim, n_sim = global_confidence(idx_sim, model)

# ---- results: only two measures for both logs -------------------------------
confidence_result = {
    "input_gconf": g_in,
    "simulated_gconf": g_sim,
    "input_mean_confidence": float(conf_in["confidence"].mean(skipna=True)),
    "simulated_mean_confidence": float(conf_sim["confidence"].mean(skipna=True)),
}

print_result(
    confidence_result,
    "OC-Declare confidence (Definitions 8 & 9)"
)

OC-Declare confidence (Definitions 8 & 9)
  input_gconf                   0.0049   
  simulated_gconf               0.0002   
  input_mean_confidence         0.4550  [0-1]   [0-1], 1 = fully conforming
  simulated_mean_confidence     0.2701  [0-1]   [0-1], 1 = fully conforming


Constraint Violations (object-centric, per-activation):

In [56]:
# ===== Constraint violations (Definition 16, RV-LTL states) =================
# needs INPUT_LOG, SIMULATED_LOG, OCDECLARE_MODEL; self-contained (repeats Def. 8 machinery)
import json, itertools, numpy as np, pandas as pd
from collections import defaultdict, Counter

EP_ACTIVATES_ON_TARGET = True     # OCDeclarediscovery writes EP arcs swapped:
                                  # "EP/DP(A->B,T): precedence(source=B, target=A)",
                                  # so Definition 8's activation s is target_activity.
# nmax is deliberately NOT enforced: the "nmax" field written by OCDeclarediscovery is
# the maximum occurrences of an activity PER SCOPE OBJECT (see _arcs_to_deco_constraints:
# "target may occur at most this often"), not Definition 8's bound on |f(E_L)|. All 35
# constraints carry nmin=1, so "nmin <= |f(E_L)|" is Definition 8 in full on that side.

# ---- unit-aware printing ----------------------------------------------------
def _unit_of(key):
    k = key.lower()
    if k.startswith(("conf", "gconf")) or k.endswith(("_confidence", "_rate", "_coverage")) \
       or k.endswith("share_permanent"):                      return "fraction"
    if k.startswith("wmape") or k.endswith("_norm"):             return "ratio"
    if k.endswith(("_hours", "_h")) or "_h_" in k:               return "hours"
    return "count" if k.split("_")[-1] in ("violations","permanent","repairable","affected",
                                           "events","objects","constraints") else "value"

_SCALES = {"fraction": "[0-1]", "ratio": "0 % = identical, unbounded",
           "hours": "[0, inf) hours", "count": "", "value": ""}

def _fmt(key, v):
    if not isinstance(v, (int, float, np.floating, np.integer)) or isinstance(v, bool): return str(v)
    if v != v: return "       —"
    u = _unit_of(key)
    if u == "fraction": return f"{v:10.4f}  [0-1]"
    if u == "ratio":    return f"{v*100:10.2f} %    "
    if u == "hours":    return f"{v:10.4f}  h    "
    if u == "count":    return f"{v:10.0f}       "
    if isinstance(v, (int, np.integer)): return f"{v:10d}       "
    return f"{v:10.4f}       "

def print_result(d, title=None):
    if title: print(title)
    for k, v in d.items():
        scale = _SCALES.get(_unit_of(k), "") if isinstance(v, (int, float, np.floating, np.integer)) else ""
        print(f"  {k:32}{_fmt(k, v)}   {scale}")

def format_table(df):
    out = df.copy()
    for c in out.columns:
        if out[c].dtype.kind in "fi": out[c] = [_fmt(c, x).strip() for x in out[c]]
    return out

# ---- index + Definition 8 (same machinery as the confidence cell) -----------
def build_index(path):
    raw = json.load(open(path))
    otype = {o["id"]: o["type"] for o in raw["objects"]}
    ev_objs, obj_events = defaultdict(lambda: defaultdict(set)), defaultdict(set)
    ev_time, act_events = {}, defaultdict(set)
    ts = pd.to_datetime([e["time"] for e in raw["events"]], utc=True, format="ISO8601")
    for e, t in zip(raw["events"], ts):
        ev_time[e["id"]] = t; act_events[e["type"]].add(e["id"])
        for r in e.get("relationships") or []:
            o = r["objectId"]; ev_objs[e["id"]][otype.get(o)].add(o); obj_events[o].add(e["id"])
    return {"ev_objs": ev_objs, "obj_events": obj_events, "ev_time": ev_time,
            "act_events": act_events, "otype": otype, "n_events": len(raw["events"])}

def constraint_roles(c, ep_on_target=EP_ACTIVATES_ON_TARGET):
    ar = c.get("arc_type", "EF").upper()
    s, t = c["source_activity"], c["target_activity"]
    if ar == "EP" and ep_on_target: return t, s, "before", ar
    return (s, t, "after", ar) if ar == "EF" else (s, t, "before", ar)

def involvement(c):
    inv = c.get("involvement_per_label") or {lab: "each" for lab in c.get("label", [])}
    low = {k: str(v).lower() for k, v in inv.items()}
    return ([k for k, v in low.items() if v == "each"],
            [k for k, v in low.items() if v == "all"],
            [k for k, v in low.items() if v == "any"])

def f_filter(idx, e, act_t, direction, each_combo, all_objs, any_types):
    required = set(each_combo) | all_objs
    if required:
        sets = sorted((idx["obj_events"][o] for o in required), key=len)
        cands = set(sets[0])
        for s in sets[1:]: cands &= s
        cands &= idx["act_events"][act_t]
    else:
        cands = set(idx["act_events"][act_t])
    te = idx["ev_time"][e]
    cands = {x for x in cands
             if (idx["ev_time"][x] > te if direction == "after" else idx["ev_time"][x] < te)}
    for ot in any_types:
        # Definition 8 applies filter^ANY unconditionally: if e has no object of this
        # type the filter admits nothing, f(E_L) is empty and the activation is violated.
        src = idx["ev_objs"][e].get(ot, set())
        cands = {x for x in cands if idx["ev_objs"][x].get(ot, set()) & src}
    return cands

# ---- Definition 16: RV-LTL states ------------------------------------------
# This cell does NOT report a satisfaction rate — that is conf_L(D), computed in the
# confidence cell (with these settings the two are numerically identical). What RV-LTL adds
# is WHICH violations are permanent vs still repairable, and WHICH objects caused them.
STATES = ["PERM_VIOL", "CURR_VIOL", "CURR_SAT", "PERM_SAT"]      # worst first

def rv_ltl_state(n, nmin, nmax, direction):
    """'before' (EP): the past is fixed -> the verdict is permanent at activation time (PT/PL).
       'after'  (EF): the count can still grow -> too few is only a CURRENT violation (CL), and
       in-bounds is only a CURRENT satisfaction (CT) while an upper bound could still be broken."""
    if direction == "before":  return "PERM_SAT" if nmin <= n <= nmax else "PERM_VIOL"
    if n > nmax:               return "PERM_VIOL"
    if n < nmin:               return "CURR_VIOL"
    return "PERM_SAT" if nmax == float("inf") else "CURR_SAT"

def violations(idx, model, max_per_constraint=None):
    """One row per violating (constraint, event, object combination) — an incident list, not a
       score. 'objects' names the object that caused it (Def. 9's advanced diagnostics)."""
    inc = []
    for c in model["constraints"]:
        act_s, act_t, direction, ar = constraint_roles(c)
        each, alls, anys = involvement(c)
        nmin = c.get("nmin", 1)
        nmax = float("inf")                     # see the note at the top of the cell
        label = f'{c.get("type")}({c["source_activity"]} -> {c["target_activity"]})'
        seen = 0
        for e in sorted(idx["act_events"][act_s], key=lambda x: idx["ev_time"][x]):
            objs = idx["ev_objs"][e]
            all_objs = set().union(*[objs.get(ot, set()) for ot in alls]) if alls else set()
            domains = [sorted(objs.get(ot, set())) for ot in each]
            # empty Each-domain: vacuously true (Def. 8)
            for combo in (itertools.product(*domains) if domains else [()]):
                n = len(f_filter(idx, e, act_t, direction, combo, all_objs, anys))
                if nmin <= n <= nmax: continue
                state = rv_ltl_state(n, nmin, nmax, direction)
                inc.append({"constraint": label, "arc": ar, "state": state,
                            "repairable": state == "CURR_VIOL",
                            "activation_activity": act_s, "counted_activity": act_t,
                            "event": e, "time": idx["ev_time"][e],
                            "objects": ", ".join(combo),
                            "object_types": ", ".join(idx["otype"].get(o, "?") for o in combo),
                            "observed": n,
                            "expected": f">= {nmin}" if nmax == float("inf") else f"[{nmin},{nmax}]"})
                seen += 1
                if max_per_constraint and seen >= max_per_constraint: break
            if max_per_constraint and seen >= max_per_constraint: break
    return pd.DataFrame(inc)

model = json.load(open(OCDECLARE_MODEL))
idx_in, idx_sim = build_index(INPUT_LOG), build_index(SIMULATED_LOG)
viol_input, viol_sim = violations(idx_in, model), violations(idx_sim, model)

def _summary(v, tag):
    n = len(v)
    return {f"{tag}_violations": n,
            f"{tag}_permanent": int((v.state == "PERM_VIOL").sum()) if n else 0,
            f"{tag}_repairable": int((v.state == "CURR_VIOL").sum()) if n else 0,
            f"{tag}_share_permanent": float((v.state == "PERM_VIOL").mean()) if n else np.nan,
            f"{tag}_constraints_affected": int(v.constraint.nunique()) if n else 0,
            f"{tag}_events_affected": int(v.event.nunique()) if n else 0,
            f"{tag}_objects_affected": int(pd.unique(v.objects).size) if n else 0}

violation_result = {**_summary(viol_input, "input"), **_summary(viol_sim, "simulated")}
print_result(violation_result, "Constraint violations  (Def. 16: permanent PL vs repairable CL, with diagnostics)")

if len(viol_sim):
    print("\nby constraint (simulated log):")
    by_c = (viol_sim.groupby(["constraint", "arc"])
            .agg(violations=("event", "size"), permanent=("state", lambda s: (s == "PERM_VIOL").sum()),
                 repairable=("state", lambda s: (s == "CURR_VIOL").sum()),
                 events=("event", "nunique"), objects=("objects", "nunique"))
            .reset_index().sort_values("violations", ascending=False))
    print(format_table(by_c).head(10).to_string(index=False))

    print("\nwhere they land (simulated log):")
    print("  by activation activity:", dict(viol_sim.activation_activity.value_counts().head(5)))
    print("  by object type of the offending object:", dict(viol_sim.object_types.value_counts().head(5)))

    print("\nfirst incidents in time:")
    print(format_table(viol_sim.nsmallest(5, "time")[["time", "constraint", "state", "objects",
                                                      "observed", "expected"]]).to_string(index=False))

viol_sim

Constraint violations  (Def. 16: permanent PL vs repairable CL, with diagnostics)
  input_violations                       692          
  input_permanent                   379.0000  [0-1]   [0-1]
  input_repairable                       313          
  input_share_permanent               0.5477  [0-1]   [0-1]
  input_constraints_affected              14          
  input_events_affected                  625          
  input_objects_affected                 609          
  simulated_violations                 20957          
  simulated_permanent              6377.0000  [0-1]   [0-1]
  simulated_repairable                 14580          
  simulated_share_permanent           0.3043  [0-1]   [0-1]
  simulated_constraints_affected          23          
  simulated_events_affected            15408          
  simulated_objects_affected           13975          

by constraint (simulated log):
                                       constraint arc violations permanent repairable events obj

,constraint,arc,state,repairable,activation_activity,counted_activity,event,time,objects,object_types,observed,expected
0,response(Book Vehicles -> Depart),EF,CURR_VIOL,True,Book Vehicles,Depart,e74,2025-01-03 08:15:54.328148+00:00,"Transport Document_2, Vehicle_1","Transport Document, Vehicle",0,>= 1
1,response(Book Vehicles -> Depart),EF,CURR_VIOL,True,Book Vehicles,Depart,e347,2025-01-08 05:19:57.918099+00:00,"Transport Document_5, Vehicle_3","Transport Document, Vehicle",0,>= 1
2,response(Book Vehicles -> Depart),EF,CURR_VIOL,True,Book Vehicles,Depart,e354,2025-01-08 06:40:33.933937+00:00,"Transport Document_6, Vehicle_4","Transport Document, Vehicle",0,>= 1
3,response(Book Vehicles -> Depart),EF,CURR_VIOL,True,Book Vehicles,Depart,e355,2025-01-08 07:05:46.718219+00:00,"Transport Document_8, Vehicle_5","Transport Document, Vehicle",0,>= 1
4,response(Book Vehicles -> Depart),EF,CURR_VIOL,True,Book Vehicles,Depart,e382,2025-01-08 08:19:28.777629+00:00,"Transport Document_7, Vehicle_6","Transport Document, Vehicle",0,>= 1
...,...,...,...,...,...,...,...,...,...,...,...,...
20952,response(Weigh -> Place in Stock),EF,CURR_VIOL,True,Weigh,Place in Stock,e29883,2026-04-01 08:17:18.540145+00:00,"Container_1938, Forklift_1095","Container, Forklift",0,>= 1
20953,response(Weigh -> Place in Stock),EF,CURR_VIOL,True,Weigh,Place in Stock,e29888,2026-04-01 08:20:14.902514+00:00,"Container_1948, Forklift_1096","Container, Forklift",0,>= 1
20954,response(Weigh -> Place in Stock),EF,CURR_VIOL,True,Weigh,Place in Stock,e29894,2026-04-01 08:25:57.071409+00:00,"Container_1949, Forklift_1097","Container, Forklift",0,>= 1
20955,response(Weigh -> Place in Stock),EF,CURR_VIOL,True,Weigh,Place in Stock,e29912,2026-04-03 09:55:08.586233+00:00,"Container_1954, Forklift_1098","Container, Forklift",0,>= 1


Coverage of Elements:

In [75]:
# ===== Coverage of the OC-Declare model by each log =========================
# needs INPUT_LOG, SIMULATED_LOG, OCDECLARE_MODEL (paths) from the selection cells
import json, itertools, numpy as np, pandas as pd
from collections import defaultdict

EP_ACTIVATES_ON_TARGET = True     # OCDeclarediscovery writes EP arcs swapped:
                                  # "EP/DP(A->B,T): precedence(source=B, target=A)",
                                  # so Definition 8's activation s is target_activity.
# nmax is deliberately NOT enforced: the "nmax" field written by OCDeclarediscovery is
# the maximum occurrences of an activity PER SCOPE OBJECT (see _arcs_to_deco_constraints:
# "target may occur at most this often"), not Definition 8's bound on |f(E_L)|. All 35
# constraints carry nmin=1, so "nmin <= |f(E_L)|" is Definition 8 in full on that side.

# ---- index + Definition 8 machinery -----------------------------------------
def build_index(path):
    raw = json.load(open(path))
    otype = {o["id"]: o["type"] for o in raw["objects"]}
    ev_objs, obj_events = defaultdict(lambda: defaultdict(set)), defaultdict(set)
    ev_time, act_events = {}, defaultdict(set)
    ts = pd.to_datetime([e["time"] for e in raw["events"]], utc=True, format="ISO8601")
    for e, t in zip(raw["events"], ts):
        ev_time[e["id"]] = t; act_events[e["type"]].add(e["id"])
        for r in e.get("relationships") or []:
            o = r["objectId"]; ev_objs[e["id"]][otype.get(o)].add(o); obj_events[o].add(e["id"])
    return {"ev_objs": ev_objs, "obj_events": obj_events, "ev_time": ev_time,
            "act_events": act_events, "otype": otype, "n_events": len(raw["events"])}

def constraint_roles(c, ep_on_target=EP_ACTIVATES_ON_TARGET):
    ar = c.get("arc_type", "EF").upper()
    s, t = c["source_activity"], c["target_activity"]
    if ar == "EP" and ep_on_target: return t, s, "before", ar
    return (s, t, "after", ar) if ar == "EF" else (s, t, "before", ar)

def involvement_types(c):
    inv = c.get("involvement_per_label") or {lab: "each" for lab in c.get("label", [])}
    low = {k: str(v).lower() for k, v in inv.items()}
    return ([k for k, v in low.items() if v == "each"],
            [k for k, v in low.items() if v == "all"],
            [k for k, v in low.items() if v == "any"])

def f_filter(idx, e, act_t, direction, each_combo, all_objs, any_types):
    required = set(each_combo) | all_objs
    if required:
        sets = sorted((idx["obj_events"][o] for o in required), key=len)
        cands = set(sets[0])
        for s in sets[1:]: cands &= s
        cands &= idx["act_events"][act_t]
    else:
        cands = set(idx["act_events"][act_t])
    te = idx["ev_time"][e]
    cands = {x for x in cands
             if (idx["ev_time"][x] > te if direction == "after" else idx["ev_time"][x] < te)}
    for ot in any_types:
        # Definition 8 applies filter^ANY unconditionally: if e has no object of this
        # type the filter admits nothing, f(E_L) is empty and the activation is violated.
        src = idx["ev_objs"][e].get(ot, set())
        cands = {x for x in cands if idx["ev_objs"][x].get(ot, set()) & src}
    return cands

# ---- the five coverage dimensions -------------------------------------------
def coverage(idx, model):
    """ELEMENTS   share of the model's activities and object types seen in the log.
       ACTIVATION share of constraints that were triggered at least once (non-vacuous).
       INVOLVEMENT how thoroughly each constraint's object scope was exercised: the share of
                   activations that actually carry an object of EVERY label type of the
                   constraint (an activation missing one binds a universal quantifier over an
                   empty set, so it holds trivially and tests nothing).
       COUNTS     boundary coverage of the cardinality interval: were n = nmin and n = nmax
                  each actually witnessed among the observed |f(E_L)| values?
       ARROWS     share of constraint relations actually traversed: at least one activation
                  with a matching counted event (source firing alone is not enough)."""
    acts_model = {a["name"] for a in model["activities"]}
    types_model = {t["name"] for t in model["object_types"]}
    acts_log = {a for a, evs in idx["act_events"].items() if evs}
    types_log = set(idx["otype"].values())
    elements = ((len(acts_model & acts_log) + len(types_model & types_log))
                / max(len(acts_model) + len(types_model), 1))

    rows = []
    for c in model["constraints"]:
        act_s, act_t, direction, ar = constraint_roles(c)
        each, alls, anys = involvement_types(c)
        labels = each + alls + anys
        nmin = c.get("nmin", 1); nmax_decl = c.get("nmax", 1)
        nmax = float("inf")                     # see the note at the top of the cell
        events = idx["act_events"][act_s]
        full_binding, seen_counts, arrow = 0, set(), False
        for e in events:
            objs = idx["ev_objs"][e]
            if not labels or all(objs.get(ot) for ot in labels): full_binding += 1
            all_objs = set().union(*[objs.get(ot, set()) for ot in alls]) if alls else set()
            domains = [sorted(objs.get(ot, set())) for ot in each]
            # empty Each-domain: vacuously true (Def. 8)
            for combo in (itertools.product(*domains) if domains else [()]):
                n = len(f_filter(idx, e, act_t, direction, combo, all_objs, anys))
                seen_counts.add(n)
                if n >= 1: arrow = True
        activated = len(events) > 0
        inv_score = full_binding/len(events) if events else np.nan
        # COUNTS follows Definition 8: the bounds constrain |f(E_L)|, so coverage asks whether
        # the observed |f(E_L)| values witnessed the interval's endpoints. n_max is instantiated
        # as infinity (see the note at the top of the cell), so only the lower bound is
        # witnessable: did some activation sit exactly AT n_min, or was it always exceeded?
        if not activated:
            cnt_score = np.nan
        else:
            bounds = [nmin in seen_counts]
            if nmax != float("inf"): bounds.append(nmax in seen_counts)
            cnt_score = sum(bounds)/len(bounds)
        rows.append({"constraint": f'{c.get("type")}({c["source_activity"]} -> {c["target_activity"]})',
                     "arc": ar, "n_activations": len(events), "activated": activated,
                     "involvement": inv_score, "nmin": nmin, "nmax_file": nmax_decl,   # the file's field, not Def-8 n_max
                     "counts": cnt_score, "arrow": arrow,
                     "observed_counts": f"{min(seen_counts)}..{max(seen_counts)}" if seen_counts else "—"})
    det = pd.DataFrame(rows)
    scores = {"Elements": elements,
              "Activation": float(det.activated.mean()),
              "Involvement": float(det.involvement.mean(skipna=True)),
              "Counts": float(det["counts"].mean(skipna=True)),
              "Arrows": float(det.arrow.mean())}
    scores = {"Overall": float(np.mean(list(scores.values()))), **scores}
    return scores, det

model = json.load(open(OCDECLARE_MODEL))
cov_input, det_input = coverage(build_index(INPUT_LOG), model)
cov_sim,   det_sim   = coverage(build_index(SIMULATED_LOG), model)

coverage_table = pd.DataFrame({"input log": cov_input, "simulated log": cov_sim})
coverage_table["difference"] = coverage_table["input log"] - coverage_table["simulated log"]
print("Model coverage — all values in [0-1], 1 = fully exercised")
print(coverage_table.to_string(float_format=lambda v: f"{v:8.4f}"))

Model coverage — all values in [0-1], 1 = fully exercised
             input log  simulated log  difference
Overall         0.9878         0.9686      0.0192
Elements        1.0000         1.0000      0.0000
Activation      1.0000         1.0000      0.0000
Involvement     0.9962         1.0000     -0.0038
Counts          0.9429         0.9000      0.0429
Arrows          1.0000         0.9429      0.0571
